In [ ]:
import json, os
from pathlib import Path
from collections import Counter

from google.colab import drive
drive.mount('/content/drive')

LABELS_ROOT = "/content/drive/MyDrive/Capstone/project/labels"

def compute_iaa(folder_path):
    counts = Counter()
    files = 0
    for fpath in Path(folder_path).glob("*.json"):
        if fpath.name == "meta.json":
          continue
        with open(fpath) as f:
            batch = json.load(f)
        for entry in batch.values():
            for status in entry["b"]["status"]:
                counts[status] += 1
        files += 1

    ab = counts["agree_boundary"]
    po = counts["psst_only"]
    wo = counts["wav2tobi_only"]
    an = counts["agree_none"]

    total_boundary = ab + po + wo
    symmetric_iaa  = ab / total_boundary if total_boundary else 0
    psst_centric   = ab / (ab + po) if (ab + po) else 0

    print(f"  Files: {files}")
    print(f"  agree_boundary: {ab:,}  psst_only: {po:,}  wav2tobi_only: {wo:,}  agree_none: {an:,}")
    print(f"  Symmetric IAA:  {symmetric_iaa:.1%}  (agree / all boundary positions)")
    print(f"  PSST-centric:   {psst_centric:.1%}  (agree / agree+psst_only)")

print("=== LibriTTS clean-100 ===")
compute_iaa(f"{LABELS_ROOT}/clean-100")

print("\n=== LibriTTS clean-360 ===")
compute_iaa(f"{LABELS_ROOT}/clean-360")

print("\n=== People's Speech ===")
compute_iaa(f"{LABELS_ROOT}/ps")

In [ ]:
for fpath in Path(f"{LABELS_ROOT}/clean-360").glob("*.json"):
    with open(fpath) as f:
        batch = json.load(f)
    for key, val in batch.items():
        if not isinstance(val, dict):
            print(f"Bad entry in {fpath.name}: key={key}, type={type(val)}")
            break
        if "b" not in val:
            print(f"Missing 'b' key in {fpath.name}: key={key}, keys={list(val.keys())}")
            break